# Error severity: NAV sensitivity and distinct accounts

Run from this repo with its `.venv`. This notebook reruns notebook 02 unchanged, reuses its full-data classifications, and writes `outputs/03_*.csv` plus a severity plot. No raw data is modified.

**Financial axis:** alternative minus current investment value, USD and basis points of each fund's reported net assets. These are indicative, investment-only scenarios; missing cash/accruals and unknown corrections prevent a claim of actual NAV misstatement or loss. NAV arithmetic uses a separate equivalent-dollar inconsistency, assuming net assets and shares are authoritative. Vendor flags use high-minus-low quote scenarios, not a verified correction.

**Breadth axis:** distinct account IDs in each defect/scenario population. NAV arithmetic uses the affected fund/share-class account cohort, explicitly distinguished from directly implicated holding accounts. Fund-wide account cohorts are also reported for holding scenarios; these are possible downstream reach, not confirmed investor harm.

Only unique-master EQUITY records with eligible, unambiguous quotes and FX are monetized. Bonds have unresolved quote/quantity basis. Generic holding scenarios exclude existing vendor spread/scale flags. Price scenarios intentionally quantify those flags. Unknowns remain blank, with coverage and reasons. No materiality thresholds or combined score are invented. Classes and scenarios overlap and must not be summed.


In [ ]:
from pathlib import Path
impact_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'manifest.csv').exists())
get_ipython().run_line_magic('run', str(impact_root / 'notebooks/02_reconcile.ipynb'))


In [ ]:
# Unique mappings; quarantine duplicate master keys instead of expanding joins.
impact_accounts = accounts.loc[~accounts.duplicated('account_id', keep=False), ['account_id','fund_id','share_class']]
impact_securities = securities.loc[~securities.duplicated('security_id', keep=False), ['security_id','asset_class','currency']].rename(columns={'currency':'master_currency'})
impact_fx = fx_rates.assign(day=normalise_date(fx_rates['rate_date']), fx=pd.to_numeric(fx_rates['rate_to_usd'], errors='coerce'))
impact_fx = impact_fx[impact_fx['day'].eq(AS_OF_DATE)]
impact_fx = impact_fx[~impact_fx.duplicated('currency',keep=False) & impact_fx['fx'].gt(0)]
impact_quotes = snapshot_quotes.copy().merge(impact_securities,on='security_id',how='left',validate='many_to_one')
impact_quotes['priority'] = pd.to_numeric(impact_quotes['vendor_priority'],errors='coerce')
impact_quotes['timestamp'] = to_time(impact_quotes['price_timestamp'])
impact_quotes = impact_quotes[impact_quotes['price_number'].gt(0) & impact_quotes['priority'].notna()
    & impact_quotes['currency'].eq(impact_quotes['master_currency'])
    & impact_quotes['timestamp'].dt.strftime('%Y-%m-%d').eq(AS_OF_DATE)]
# A duplicate vendor quote is ambiguous, even if one row looks usable.
impact_duplicate_quotes = snapshot_quotes[snapshot_quotes.duplicated(['security_id','vendor'],keep=False)][['security_id','vendor']].drop_duplicates()
impact_quotes = impact_quotes.merge(impact_duplicate_quotes.assign(duplicate=True),on=['security_id','vendor'],how='left',validate='many_to_one')
impact_quotes = impact_quotes[~impact_quotes['duplicate'].eq(True)]
impact_best = impact_quotes[impact_quotes['priority'].eq(impact_quotes.groupby('security_id')['priority'].transform('min'))]
impact_best = impact_best[~impact_best.duplicated('security_id',keep=False)]
impact_price = impact_securities.merge(impact_best[['security_id','price_number','vendor']],on='security_id',how='left',validate='one_to_one')
impact_price = impact_price.merge(impact_fx[['currency','fx']].rename(columns={'currency':'master_currency'}),on='master_currency',how='left',validate='many_to_one')
impact_price.loc[impact_price['master_currency'].eq(REPORTING_CURRENCY),'fx']=1.0
impact_price = impact_price.merge(vendor_check[['security_id','result']].drop_duplicates(),on='security_id',how='left',validate='one_to_one')
impact_price['unit_usd'] = impact_price['price_number'] * impact_price['fx']
impact_price['valuation_reason'] = np.select([
    ~impact_price['asset_class'].eq('EQUITY'), impact_price['price_number'].isna(), impact_price['fx'].isna(),
    ~impact_price['result'].eq('pass')],
    ['quote/quantity basis not verified for asset class','no unique eligible snapshot quote','no unique positive snapshot FX',
     'vendor dispersion or insufficient independent quotes'],default='eligible equity scenario')
impact_price.loc[impact_price['valuation_reason'].ne('eligible equity scenario'),'unit_usd']=np.nan

impact_ledger = ledger_nav.assign(day=normalise_date(ledger_nav['as_of_date']), na=pd.to_numeric(ledger_nav['net_assets'],errors='coerce'))
impact_ledger = impact_ledger[impact_ledger['day'].eq(AS_OF_DATE)].copy()
impact_bad_funds = set(impact_ledger.loc[impact_ledger.duplicated(['fund_id','share_class'],keep=False) | impact_ledger['na'].isna(),'fund_id'])
impact_funds = impact_ledger.groupby('fund_id',as_index=False).agg(fund_net_assets_usd=('na',lambda x:x.sum(min_count=1)))
impact_funds.loc[impact_funds['fund_id'].isin(impact_bad_funds) | impact_funds['fund_net_assets_usd'].le(0),'fund_net_assets_usd']=np.nan
impact_details, impact_members, impact_specs = [], [], []


def impact_add(name, frame, interpretation, evidence='unresolved', zero=False, price_span=False):
    """A scenario has one financial row per account/security key, not per raw trade row."""
    frame=frame.copy()
    assert not frame.duplicated(KEY).any(), name
    impact_specs.append(dict(scenario=name, interpretation=interpretation, evidence=evidence, account_scope='directly implicated account IDs'))
    d=frame.merge(impact_accounts,on='account_id',how='left',validate='many_to_one')
    d=d.merge(impact_price,on='security_id',how='left',validate='many_to_one')
    d['valuation_reason']=d['valuation_reason'].fillna('missing or nonunique security master')
    d['delta_usd']=d['delta_units']*d['unit_usd']
    d.loc[d['delta_units'].eq(0),'delta_usd']=0.0
    d.loc[d['delta_units'].eq(0),'valuation_reason']='zero investment-unit change under this scenario'
    if price_span:
        span=impact_quotes.groupby('security_id',as_index=False).agg(low=('price_number','min'),high=('price_number','max'),vendors=('vendor','nunique'))
        d=d.merge(span,on='security_id',how='left',validate='many_to_one')
        eligible=d['asset_class'].eq('EQUITY') & d['fx'].notna() & d['vendors'].ge(2)
        d['delta_usd']=(d['quantity']*(d['high']-d['low'])*d['fx']).where(eligible)
        d['valuation_reason']=np.where(eligible,'equity high-minus-low vendor scenario','quote basis, quotes or FX unavailable')
    if zero:
        d['delta_usd']=0.0
        d['valuation_reason']='zero investment-unit effect under this reading; not evidence the record is valid'
    d['scenario']=name
    impact_details.append(d[['scenario',*KEY,'fund_id','share_class','delta_units','delta_usd','valuation_reason']])
    impact_members.append(d[['scenario','account_id','fund_id','share_class']].drop_duplicates())

# Negative rows: isolate the effect of the sign interpretation; do not repair the book.
impact_neg=sign_conflict[KEY].merge(negative_rows.groupby(KEY,as_index=False).agg(delta_units=('quantity_used',lambda x:-2*x.sum())),on=KEY,how='left',validate='one_to_one').fillna({'delta_units':0.0})
impact_add('Negative quantities: wrong-sign scenario',impact_neg,'Flip the signed contribution of active negative rows; all other treatments fixed')
impact_add('Negative quantities: wrong-type scenario',impact_neg.assign(delta_units=0.0),'Change BUY/SELL interpretation while retaining its signed holding contribution',zero=True)
# Blanks: isolate derived-versus-zero units only on keys already selected by notebook 02.
impact_blank=BLANK_OMITTED[KEY].merge(derived_vs_book[KEY+['rebuilt_quantity']],on=KEY,validate='one_to_one').merge(
    zero_vs_book[KEY+['rebuilt_quantity']].rename(columns={'rebuilt_quantity':'zero_quantity'}),on=KEY,validate='one_to_one')
impact_blank['delta_units']=impact_blank['rebuilt_quantity']-impact_blank['zero_quantity']
impact_add('Blank quantity: derived-units scenario',impact_blank[KEY+['delta_units']],'Gross amount / price derivation versus blank-as-zero; source confirmation still needed')
# Reconciliation classes are disjoint by construction, but may overlap the flags above.
for impact_class, impact_group in three_way[three_way['is a break'].eq('yes')].groupby('main_class',sort=True):
    impact_frame=impact_group[KEY].copy()
    impact_frame['delta_units']=(impact_group['rebuilt_quantity']-impact_group['book_quantity']).to_numpy()
    impact_add('Reconciliation: '+impact_class,impact_frame,'Main-chain rebuild minus book; benchmark sensitivity, not a verified correction')
# Existence gaps: no price and no authoritative corrected units, so retain unknown dollars.
for impact_name,impact_mask in [
    ('Missing security master / spin-off target',~positions['security_id'].isin(securities['security_id'])),
    ('Held security with no price',~positions['security_id'].isin(prices['security_id']))]:
    impact_add(impact_name,positions.loc[impact_mask,KEY].assign(delta_units=np.nan),'Observed affected holdings; entitlement and valuation remain unknown','confirmed')
impact_add('Book holding absent from custodian',book_vs_custodian.loc[book_vs_custodian['side'].eq('only in positions'),KEY].assign(delta_units=np.nan),
    'Missing confirmation is not zero custody; no corrected quantity is known')
impact_add('Closed flat positions: cleared',three_way.loc[three_way['pattern'].eq('rebuild only, nets to zero'),KEY].assign(delta_units=0.),
    'Closed positions omitted by both sources under the documented reporting convention','cleared',zero=True)
for impact_result in ['spread above threshold','scale flag (ratio near 100)']:
    impact_flagged=positions[positions['security_id'].isin(vendor_check.loc[vendor_check['result'].eq(impact_result),'security_id'])][KEY+['quantity']].copy()
    impact_flagged['quantity']=pd.to_numeric(impact_flagged['quantity'],errors='coerce')
    impact_flagged['delta_units']=np.nan
    impact_add('Vendor quotes: '+impact_result,impact_flagged,'High quote versus low quote at the same holding; neither vendor is assumed correct',price_span=True)

# NAV arithmetic: dollar equivalent is kept once per ledger row, never repeated once per account.
impact_nav=nav_check[nav_check['result'].eq('fail') & nav_check['as_of_day'].eq(pd.Timestamp(AS_OF_DATE))].copy()
impact_nav_name='NAV arithmetic: net-assets/shares versus published NAV'
impact_specs.append(dict(scenario=impact_nav_name,interpretation='net_assets - nav_per_share * shares; equivalent inconsistency, not a known correction',evidence='confirmed',account_scope='accounts mapped to failing fund/share-class rows'))
impact_nav['delta_usd']=impact_nav['net_assets_number']-impact_nav['nav_per_share_number']*impact_nav['shares_outstanding_number']
impact_nav['scenario']=impact_nav_name
impact_nav['account_id']=''
impact_nav['security_id']=''
impact_nav['delta_units']=np.nan
impact_nav['valuation_reason']='arithmetic dollar equivalent; net assets and shares assumed authoritative'
impact_nav.loc[impact_nav['repeated key'],'delta_usd']=np.nan
impact_details.append(impact_nav[['scenario',*KEY,'fund_id','share_class','ledger_id','delta_units','delta_usd','valuation_reason']])
impact_nav_cohort=impact_accounts.merge(impact_nav[['fund_id','share_class']].drop_duplicates(),on=['fund_id','share_class'],how='inner',validate='many_to_one').assign(scenario=impact_nav_name)
impact_members.append(impact_nav_cohort[['scenario','account_id','fund_id','share_class']])
impact_detail=pd.concat(impact_details,ignore_index=True)
impact_membership=pd.concat(impact_members,ignore_index=True).drop_duplicates()
impact_spec=pd.DataFrame(impact_specs)
print('Scenarios:',len(impact_spec),'| financial detail rows:',len(impact_detail))


In [ ]:
# Aggregate within funds first. Never add basis points with different fund denominators.
def impact_aggregate(detail):
    return detail.groupby(['scenario','fund_id'],as_index=False,dropna=False).agg(
        signed_delta_usd=('delta_usd',lambda s:s.sum(min_count=1)),
        gross_delta_usd=('delta_usd',lambda s:s.abs().sum(min_count=1)),
        detail_rows=('delta_usd','size'),valued_rows=('delta_usd','count'))

impact_by_fund=impact_aggregate(impact_detail).merge(impact_funds,on='fund_id',how='left',validate='many_to_one')
impact_by_fund['signed_delta_bps']=impact_by_fund['signed_delta_usd']/impact_by_fund['fund_net_assets_usd']*1e4
impact_by_fund['gross_delta_bps']=impact_by_fund['gross_delta_usd']/impact_by_fund['fund_net_assets_usd']*1e4
impact_by_fund['valuation_complete']=impact_by_fund['valued_rows'].eq(impact_by_fund['detail_rows'])
impact_fund_counts=impact_membership.groupby(['scenario','fund_id'],as_index=False,dropna=False).agg(distinct_accounts=('account_id','nunique'))
impact_by_fund=impact_by_fund.merge(impact_fund_counts,on=['scenario','fund_id'],how='left',validate='one_to_one')
impact_summaries=[]
for impact_name,impact_group in impact_detail.groupby('scenario',sort=False):
    impact_m=impact_membership[impact_membership['scenario'].eq(impact_name)]
    impact_f=impact_by_fund[impact_by_fund['scenario'].eq(impact_name)]
    impact_fund_ids=impact_m['fund_id'].dropna().unique()
    impact_summaries.append(dict(scenario=impact_name,financial_rows=len(impact_group),
        distinct_accounts=impact_m['account_id'].nunique(),
        accounts_without_fund_mapping=impact_m.loc[impact_m['fund_id'].isna(),'account_id'].nunique(),
        affected_funds=len(impact_fund_ids),fund_wide_account_cohort=impact_accounts.loc[impact_accounts['fund_id'].isin(impact_fund_ids),'account_id'].nunique(),
        valued_rows=int(impact_group['delta_usd'].notna().sum()),unknown_rows=int(impact_group['delta_usd'].isna().sum()),
        valuation_coverage=impact_group['delta_usd'].notna().mean(),
        signed_delta_usd_valued_subset=impact_group['delta_usd'].sum(min_count=1),
        gross_delta_usd_valued_subset=impact_group['delta_usd'].abs().sum(min_count=1),
        max_abs_fund_net_bps_valued_subset=impact_f['signed_delta_bps'].abs().max(),
        max_fund_gross_bps_valued_subset=impact_f['gross_delta_bps'].max(),
        funds_without_NAV_denominator=int((impact_f['fund_id'].notna() & impact_f['fund_net_assets_usd'].isna()).sum())))
impact_summary=impact_spec.merge(pd.DataFrame(impact_summaries),on='scenario',how='left',validate='one_to_one')
# A scenario with no affected records counts 0 rows, accounts and funds; its dollar measures stay unknown (blank).
impact_counts=['financial_rows','distinct_accounts','accounts_without_fund_mapping','affected_funds',
               'fund_wide_account_cohort','valued_rows','unknown_rows','funds_without_NAV_denominator']
impact_summary[impact_counts]=impact_summary[impact_counts].fillna(0).astype(int)
impact_summary['account_ids_in_extract']=pd.concat([accounts['account_id'],positions['account_id'],transactions['account_id']]).nunique()
impact_summary['account_share_of_extract']=impact_summary['distinct_accounts']/impact_summary['account_ids_in_extract']
# Checks catch aggregation bugs, offsets and accidental orphan drops.
assert len(impact_detail)==len(impact_detail.drop_duplicates(['scenario','account_id','security_id','ledger_id']))
assert (impact_summary['valued_rows']+impact_summary['unknown_rows']).eq(impact_summary['financial_rows']).all()
assert impact_summary['distinct_accounts'].le(impact_summary['account_ids_in_extract']).all()
assert np.isclose(impact_detail['delta_usd'].abs().sum(),impact_by_fund['gross_delta_usd'].sum())
assert np.isclose(impact_detail['delta_usd'].sum(),impact_by_fund['signed_delta_usd'].sum())
# Synthetic checks: opposite errors must net within a fund while remaining visible gross; unknowns stay unknown.
impact_toy=pd.DataFrame({'scenario':['x','x','x','x'],'fund_id':['a','a','b',None],'delta_usd':[100.,-100.,np.nan,30.]})
impact_toy_result=impact_aggregate(impact_toy)
assert impact_toy_result.loc[impact_toy_result['fund_id'].eq('a'),'signed_delta_usd'].iloc[0]==0
assert impact_toy_result.loc[impact_toy_result['fund_id'].eq('a'),'gross_delta_usd'].iloc[0]==200
assert impact_toy_result.loc[impact_toy_result['fund_id'].eq('b'),'signed_delta_usd'].isna().all()
assert impact_toy_result['detail_rows'].sum()==4
pd.testing.assert_frame_equal(impact_aggregate(impact_detail),impact_aggregate(impact_detail.sample(frac=1,random_state=7)),check_exact=False)
for impact_file,impact_data in [('severity',impact_summary),('fund_impact',impact_by_fund),('impact_detail',impact_detail),('account_membership',impact_membership)]:
    impact_data.to_csv(OUTPUTS / f'03_{impact_file}.csv',index=False)
display(impact_summary[['scenario','distinct_accounts','affected_funds','gross_delta_usd_valued_subset','max_abs_fund_net_bps_valued_subset','valuation_coverage']])


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator
from matplotlib.patches import FancyBboxPatch

# Presentation only: all values are read from the existing severity summary.
impact_design = [
    ('A', 'Negative quantities: wrong-sign scenario', 'Negative quantity', 'Wrong-sign scenario', '#2563EB'),
    ('B', 'Reconciliation: rebuild is the odd one out: counts duplicate copies', 'Duplicate trades', 'Book + custodian count copies', '#7856B5'),
    ('C', 'Reconciliation: rebuild is the odd one out: double-offsets REVERSED', 'Reversal treatment', 'Book + custodian double-offset', '#087E8B'),
    ('D', 'Blank quantity: derived-units scenario', 'Missing trade quantity', 'Derived-units scenario', '#C27A28'),
    ('E', 'Vendor quotes: spread above threshold', 'Vendor price spread', 'High-quote vs. low-quote scenario', '#C34B71'),
    ('F', 'NAV arithmetic: net-assets/shares versus published NAV', 'NAV arithmetic', 'Equivalent-dollar inconsistency', '#344C67'),
    ('G', 'Negative quantities: wrong-type scenario', 'Negative quantity', 'Wrong-type scenario; zero unit effect', '#6F7C8D'),
    ('H', 'Closed flat positions: cleared', 'Closed positions', 'Cleared reporting convention', '#859387'),
]
impact_lookup=impact_summary.set_index('scenario',verify_integrity=True)
# A scenario exists only when its break class occurs in the data: plot the ones this run produced.
impact_design=[row for row in impact_design if row[1] in impact_lookup.index]
impact_ink='#192A40'; impact_muted='#637489'; impact_grid='#E7EDF4'
with plt.rc_context({'font.family':'DejaVu Sans','font.size':11,'axes.unicode_minus':False}):
    fig=plt.figure(figsize=(18,10.5),facecolor='#F7F9FC')
    fig.text(.055,.935,'ERROR SEVERITY',fontsize=11,fontweight='bold',color='#3564A8')
    fig.text(.055,.887,'Financial exposure × account reach',fontsize=28,fontweight='bold',color=impact_ink)
    fig.text(.055,.848,'Selected scenarios · valuation snapshot '+AS_OF_DATE+' · indicative sensitivity, not verified NAV loss',fontsize=12,color=impact_muted)

    ax=fig.add_axes([.075,.345,.45,.445],facecolor='white')
    ax.set_xscale('log')
    impact_positive=impact_summary.loc[impact_summary['scenario'].isin([r[1] for r in impact_design]),'max_abs_fund_net_bps_valued_subset'].dropna()
    impact_ymax=max(float(impact_positive.max())*1.15 if len(impact_positive) else 0.,100.)   # no valued scenario: default scale
    ax.set_xlim(25,50000); ax.set_ylim(-impact_ymax*.055,impact_ymax)
    ax.set_xticks([50,100,500,1000,5000,10000,30000])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_: f'{v/1000:g}k' if v>=1000 else f'{v:g}'))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'{v:,.0f}'))
    ax.tick_params(axis='both',length=0,labelsize=10,pad=9,colors=impact_muted)
    ax.grid(axis='y',color=impact_grid,zorder=0)
    ax.grid(axis='x',color=impact_grid,alpha=.55,zorder=0)
    for edge in ax.spines.values(): edge.set_visible(False)
    ax.axhline(0,color='#CED8E4',linewidth=1,zorder=1)
    ax.set_xlabel('Distinct accounts · logarithmic scale',fontsize=12,labelpad=14,color=impact_ink)
    ax.set_ylabel('Largest fund sensitivity · bps',fontsize=12,labelpad=12,color=impact_ink)
    fig.text(.075,.805,'FINANCIAL MAGNITUDE',fontsize=10,fontweight='bold',color=impact_muted)
    for code,name,title,subtitle,color in impact_design:
        row=impact_lookup.loc[name]
        y=row['max_abs_fund_net_bps_valued_subset']
        if pd.notna(y):
            ax.scatter(row['distinct_accounts'],y,s=360,color=color,edgecolors='white',linewidths=2,zorder=4)
            ax.text(row['distinct_accounts'],y,code,color='white',fontsize=10,fontweight='bold',ha='center',va='center',zorder=5)

    # A matching letter connects each plotted point to its exact values; no labels collide in the scatter.
    panel=fig.add_axes([.575,.318,.38,.49]); panel.axis('off')
    panel.text(0,1,'SCENARIO',fontsize=10,fontweight='bold',color=impact_muted)
    for x,label in [(.61,'ACCOUNTS'),(.80,'BPS'),(.99,'VALUED')]:
        panel.text(x,1,label,ha='right',fontsize=9,fontweight='bold',color=impact_muted)
    for i,(code,name,title,subtitle,color) in enumerate(impact_design):
        row=impact_lookup.loc[name]; y=.916-i*.113
        panel.axhline(y-.071,color=impact_grid,linewidth=.8)
        panel.scatter(.023,y,s=245,color=color,clip_on=False)
        panel.text(.023,y,code,color='white',fontsize=9,fontweight='bold',ha='center',va='center')
        panel.text(.078,y+.008,title,fontsize=11,fontweight='bold',color=impact_ink,va='bottom')
        panel.text(.078,y-.035,subtitle,fontsize=8.4,color=impact_muted,va='center')
        panel.text(.61,y,f"{int(row['distinct_accounts']):,}",ha='right',va='center',fontsize=11,color=impact_ink)
        bps=row['max_abs_fund_net_bps_valued_subset']
        panel.text(.80,y,'—' if pd.isna(bps) else f'{bps:,.1f}',ha='right',va='center',fontsize=11,fontweight='bold',color=impact_ink)
        panel.text(.99,y,f"{row['valuation_coverage']:.0%}",ha='right',va='center',fontsize=11,color=impact_muted)
    panel.set_xlim(-.015,1.015); panel.set_ylim(-.03,1.035)

    fig.text(.075,.268,'How to read it',fontsize=10,fontweight='bold',color=impact_ink)
    fig.text(.075,.244,'Higher = larger modeled fund effect. Further right = more accounts. 100 bps = 1% of fund net assets.',fontsize=10,color=impact_muted)
    fig.text(.075,.222,'BPS uses the largest absolute net fund effect on the valued subset. VALUED is scenario-wide record coverage.',fontsize=10,color=impact_muted)

    fig.text(.055,.181,'FINANCIAL IMPACT STILL UNKNOWN',fontsize=10,fontweight='bold',color=impact_muted)
    impact_unknown_cards=[
        ('Missing security master / spin-off target','Missing master / no price','Same holdings; count this cohort once'),
        ('Book holding absent from custodian','No custodian confirmation','Missing confirmation is not zero custody'),
        ('Vendor quotes: scale flag (ratio near 100)','100× vendor scale flag','Bond quote basis remains unresolved')]
    for i,(name,label,note) in enumerate(impact_unknown_cards):
        left=.055+i*.305
        fig.add_artist(FancyBboxPatch((left,.077),.288,.084,boxstyle='round,pad=0.009,rounding_size=0.008',transform=fig.transFigure,facecolor='#EBF0F6',edgecolor='none',zorder=0))
        fig.text(left+.010,.134,label,fontsize=11,fontweight='bold',color=impact_ink)
        fig.text(left+.010,.108,f"{int(impact_lookup.loc[name,'distinct_accounts']):,} accounts",fontsize=13,fontweight='bold',color=impact_ink)
        fig.text(left+.010,.087,note,fontsize=9,color=impact_muted)
    fig.text(.055,.038,'Scenarios overlap; do not sum. Account counts are direct except NAV arithmetic, which uses the fund/share-class cohort.',fontsize=9,color=impact_muted)
    fig.text(.055,.018,'Partial valuation excludes unresolved inputs. Unknown financial impact is not zero. The complete scenario list remains in 03_severity.csv.',fontsize=9,color=impact_muted)
    fig.savefig(OUTPUTS/'03_severity.png',dpi=200,facecolor=fig.get_facecolor())
    fig.savefig(OUTPUTS/'03_severity.pdf',facecolor=fig.get_facecolor())
    plt.show()
